In [1]:
import torch
import torch.nn as nn 
import torchmetrics
import optuna

/home/damian/test/test/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Ex 13

In [2]:
x = torch.tensor(1.2, dtype=torch.float32, requires_grad=True)
y = torch.tensor(3.4, dtype=torch.float32, requires_grad=True)

f = (x**2*y).sin()
f.backward()

print(x.grad, y.grad)

tensor(1.4899) tensor(0.2629)


### Ex 14

In [3]:
def Dense(layer_1, layer_2):
    module = nn.Sequential(nn.Linear(layer_1, layer_2), nn.ReLU())
    return module

### Ex 15

In [4]:

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device='cuda'

covtype = fetch_covtype()
X, y = covtype.data, covtype.target
y = y - 1

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=52, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=52, stratify=y_train)


X_train = torch.FloatTensor(X_train) #.to(device)
X_test = torch.FloatTensor(X_test) #.to(device)
X_valid = torch.FloatTensor(X_valid) #.to(device)
y_train = torch.LongTensor(y_train)#.to(device)
y_test = torch.LongTensor(y_test) #.to(device)
y_valid = torch.LongTensor(y_valid) #.to(device)


In [13]:
from torch.utils.data import DataLoader, TensorDataset


train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=True)

In [14]:
def train_model(model, optimizer, criterion, data_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
           
        mean_loss = total_loss/len(data_loader) 
       
        print(f"epoch: {epoch} loss: {mean_loss}")

In [15]:
X_instance, y_instance = TensorDataset(X_train, y_train)[1]
X_instance.shape

torch.Size([54])

In [16]:
model_1 = nn.Sequential(Dense(54, 300), Dense(300, 250), nn.Linear(250, 7))
model_1 = model_1.to('cuda', non_blocking=True)
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.01)
xentropy = torch.nn.CrossEntropyLoss()
mse = torch.nn.MSELoss()

train_model(model_1, optimizer, xentropy, train_loader, 3)

epoch: 0 loss: 0.7213552971034755
epoch: 1 loss: 0.5973578403101097
epoch: 2 loss: 0.5601471067603208


In [17]:
def eval_model(model, data_loader, metrics):
    model.eval()
    metrics.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metrics.update(y_pred, y_batch)
            
    return metrics.compute()
        
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
print(eval_model(model_1, valid_loader, accuracy))

tensor(0.7624, device='cuda:0')


In [18]:
model_besttt = nn.Sequential(Dense(54, 200), Dense(200, 100), Dense(100, 50), nn.Linear(50, 7)).to('cuda', non_blocking=True)
optimizer = torch.optim.SGD(params=model_besttt.parameters(), lr=0.005)
xentropy = nn.CrossEntropyLoss()

train_model(model_besttt, optimizer, xentropy, train_loader, 20)

epoch: 0 loss: 0.899207605688487
epoch: 1 loss: 0.6508200819839854
epoch: 2 loss: 0.6091337027042627
epoch: 3 loss: 0.5784559272802793
epoch: 4 loss: 0.5543368079436254
epoch: 5 loss: 0.5344935810323879
epoch: 6 loss: 0.5168614620197006
epoch: 7 loss: 0.5017438299416717
epoch: 8 loss: 0.48650501644605204
epoch: 9 loss: 0.47352339315539566
epoch: 10 loss: 0.4614748324493064
epoch: 11 loss: 0.4511731871727657
epoch: 12 loss: 0.44108588778147945
epoch: 13 loss: 0.4325033895013624
epoch: 14 loss: 0.42348424550783736
epoch: 15 loss: 0.4161265888982992
epoch: 16 loss: 0.409114835277069
epoch: 17 loss: 0.40138515921072715
epoch: 18 loss: 0.3947984980949127
epoch: 19 loss: 0.3888840611169515


In [19]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
print(eval_model(model_besttt, valid_loader, accuracy))

tensor(0.8187, device='cuda:0')


In [ ]:
def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int('n_hidden', 30, 300)
    
    model = nn.Sequential(Dense(54, n_hidden), Dense(n_hidden, n_hidden), Dense(n_hidden, n_hidden), nn.Linear(n_hidden, 7)).to('cuda')
    optimizer = torch.optim.SGD(params=model.parameters(), lr=lr)
    xentropy = nn.CrossEntropyLoss()
    
    train_model(model, optimizer, xentropy, train_loader, 20)
    
    accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
    valid_accuracy = eval_model(model, valid_loader, accuracy)
    
    return valid_accuracy

In [ ]:
torch.manual_seed(52)
sampler = optuna.samplers.TPESampler(seed=52)

study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=7)

[I 2026-02-20 13:10:16,697] A new study created in memory with name: no-name-f136db46-ff32-43ad-aa9a-2e90f2d5abf6
[W 2026-02-20 13:10:21,485] Trial 0 failed with parameters: {'learning_rate': 0.01960836411250069, 'n_hidden': 37} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/damian/test/test/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_120331/1634549671.py", line 9, in objective
    train_model(model, optimizer, xentropy, train_loader, 20)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_120331/2616798540.py", line 10, in train_model
    loss.backward()
    ~~~~~~~~~~~~~^^
  File "/home/damian/test/test/lib/python3.14/site-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^

KeyboardInterrupt: 

In [ ]:
model_best = nn.Sequential(Dense(54, 143), Dense(143, 143), Dense(143, 143), nn.Linear(143, 7)).to('cuda')
optimizer = torch.optim.SGD(params=model_best.parameters(), lr=0.0114768)
xentropy = nn.CrossEntropyLoss()

train_model(model_best, optimizer, xentropy, train_loader, 150)

KeyboardInterrupt: 

In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
eval_model(model_best, valid_loader, accuracy)

tensor(0.9173, device='cuda:0')

In [ ]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to('cuda')
eval_model(model_best, test_loader, accuracy)

tensor(0.9192, device='cuda:0')

In [ ]:

model_besttt = nn.Sequential(Dense(54, 200), Dense(200, 100), Dense(100, 50), nn.Linear(50, 7)).to('cuda')
optimizer = torch.optim.SGD(params=model_besttt.parameters(), lr=0.005)
xentropy = nn.CrossEntropyLoss()

train_model(model_besttt, optimizer, xentropy, train_loader, 20)
#eval_model(model_best2, valid_loader, accuracy)

RuntimeError: Pin memory thread exited unexpectedly